<b>Macroeconomic and Tourism Impact Analysis: 2010 FIFA World Cup</b><br>
Case Study: South Africa vs. Morocco (Control)<br>
This notebook evaluates the "temporary-boost hypothesis" surrounding mega-sporting events. It analyzes whether hosting the 2010 FIFA World Cup provided South Africa with sustained economic development or merely a short-term fiscal spike, using Morocco as a baseline control country for comparative analysis.<br><br>

📋 <b>Project Overview & Methodology</b><br>
Data Collection: Automatically fetches macroeconomic and tourism data from the World Bank API and government gross debt data from the IMF API.<br>

Web Scraping: Uses BeautifulSoup to scrape official assets (such as event logos) directly from Wikipedia.<br>

Comparative Analysis: Joins, aggregates, and calculates statistical percentage changes between two primary eras: Pre-World Cup (2005–2009) and Post-World Cup (2010–2015).<br>
<br>
📊 <b>Tracked Indicators</b><br>
The notebook compares South Africa (ZAF) and Morocco (MAR) across several core metrics:<br>

Economic Health: Real GDP, GDP Growth Rate, and Inflation (CPI).<br>

Labor Market: Unemployment Rates.<br>

Fiscal Position: Government Gross Debt (as a % of GDP).<br>

Tourism Performance: International Tourist Arrivals vs. International Tourism Receipts (Revenue).<br>

In [6]:
# Import libraries


# pandas and numpy support data manipulation and calculations
import pandas as pd
import numpy as np
# requests retrieves data from the World Bank and IMF APIs
import requests

# Display large numbers with commas and two decimal places
pd.set_option("display.float_format", "{:,.2f}".format)


In [7]:
# World Bank function
# Retrieve one indicator from the World Bank API and prepare it for analysis
def get_worldbank_indicator(country, indicator, metric_name):

    # Build the API request using the country and indicator codes
    url_worldbank = f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}?format=json&per_page=100"

    # Request the data and stop if the API returns an error
    response = requests.get(url_worldbank)
    response.raise_for_status()

    # Convert the API response into a two-column DataFrame
    data = response.json()

    df_country = pd.DataFrame(data[1])

    df_country = df_country[["date", "value"]]

    df_country = df_country.rename(columns={
        "date": "year",
        "value": metric_name
    })

    # Convert year to an integer and keep the 2005-2015 study period
    df_country["year"] = df_country["year"].astype(int)

    df_country = df_country[df_country["year"].between(2005, 2015)]

    return df_country.sort_values("year").reset_index(drop=True)

In [8]:
# IMF API function
# Retrieve one indicator from the IMF DataMapper API
def get_IMF_indicator(country, indicator, metric_name):

    # Build and send the request for the selected IMF indicator
    url_IMF = f"https://www.imf.org/external/datamapper/api/v2/{indicator}"

    response = requests.get(url_IMF)
    response.raise_for_status()

    data = response.json()

    # Select the year-value dictionary for the requested country
    country_data = data["values"][indicator][country]

    df_IMF_country = pd.DataFrame(
        country_data.items(),
        columns=["year", metric_name]
    )

    # Convert year to an integer and keep the 2005-2015 study period
    df_IMF_country["year"] = df_IMF_country["year"].astype(int)

    df_IMF_country = df_IMF_country[
        df_IMF_country["year"].between(2005, 2015)
    ]

    return df_IMF_country.sort_values("year").reset_index(drop=True)

In [15]:
# Build country's dataset
def build_country_dataset(wb_code, imf_code, country_name):

    # Actual GDP measured in current US dollars
    gdp = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.CD",
        "gdp_current_usd"
    )

    # Annual GDP growth rate expressed as a percentage
    gdp_growth = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.KD.ZG",
        "gdp_growth_pct"
    )

    # Annual inflation rate expressed as a percentage
    inflation = get_worldbank_indicator(
        wb_code,
        "FP.CPI.TOTL.ZG",
        "inflation_pct"
    )

    # Unemployment as a percentage of the labour force
    unemployment = get_worldbank_indicator(
        wb_code,
        "SL.UEM.TOTL.ZS",
        "unemployment_pct"
    )

    # International tourism revenue measured in current US dollars
    tourism_receipts = get_worldbank_indicator(
        wb_code,
        "ST.INT.RCPT.CD",
        "tourism_revenue_current_usd"
    )

    # International tourist arrivals

    tourist_arrivals = get_worldbank_indicator(
        wb_code,
        "ST.INT.ARVL",
        "tourist_arrivals"
    )

    # Government debt expressed as a percentage of GDP
    government_debt = get_IMF_indicator(
        imf_code,
        "GGXWDG_NGDP",
        "government_debt_pct_gdp"
    )

    # Merge all indicators by year; outer joins preserve incomplete years
    country_df = (
        gdp
        .merge(gdp_growth, on="year", how="outer")
        .merge(inflation, on="year", how="outer")
        .merge(unemployment, on="year", how="outer")
        .merge(tourism_receipts, on="year", how="outer")
        .merge(tourist_arrivals, on="year", how="outer")
        .merge(government_debt, on="year", how="outer")
    )

    # Calculate how much tourism revenue contributes relative to GDP
    country_df["tourism_revenue_pct_gdp"] = (
        country_df["tourism_revenue_current_usd"]
        / country_df["gdp_current_usd"]
        * 100
    )

    # Place the country name immediately after the year column
    country_df.insert(1, "country", country_name)

    return country_df

In [16]:
# Using South Africa as host and Morocco as control

# Build the individual country datasets using their API country codes
southafrica_df = build_country_dataset("ZAF", "ZAF", "South Africa")
morocco_df = build_country_dataset("MAR", "MAR", "Morocco")

# Stack both countries into one analysis-ready table
southafrica_morocco_df = pd.concat(
    [southafrica_df, morocco_df],
    ignore_index=True
)

# Organize the rows by country and year
southafrica_morocco_df = southafrica_morocco_df.sort_values(
    ["country", "year"]
).reset_index(drop=True)

southafrica_morocco_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,tourist_arrivals,government_debt_pct_gdp,tourism_revenue_pct_gdp
0,2005,Morocco,"68,852,658,068.67",3.19,0.98,11.01,"5,426,000,000.00","6,077,000.00",54.80,7.88
1,2006,Morocco,"75,883,823,300.85",7.79,3.28,9.67,"6,900,000,000.00","6,777,000.00",50.60,9.09
2,2007,Morocco,"86,947,913,286.73",3.44,2.04,9.56,"8,307,000,000.00","7,701,000.00",47.10,9.55
3,2008,Morocco,"101,822,906,949.06",5.68,3.71,9.57,"8,885,000,000.00","8,209,000.00",42.00,8.73
4,2009,Morocco,"101,154,952,240.88",3.75,0.97,8.96,"7,980,000,000.00","8,661,000.00",42.60,7.89
5,2010,Morocco,"100,865,329,473.44",3.50,0.99,9.09,"8,176,000,000.00","9,752,000.00",45.30,8.11
6,2011,Morocco,"110,080,631,332.38",5.52,0.91,8.91,"9,101,000,000.00","9,784,000.00",48.60,8.27
7,2012,Morocco,"106,937,392,311.13",3.06,1.29,8.99,"8,491,000,000.00","9,830,000.00",52.30,7.94
8,2013,Morocco,"115,739,287,305.08",4.12,1.88,9.23,"8,201,000,000.00","10,349,000.00",57.10,7.09
9,2014,Morocco,"119,130,841,411.66",2.72,0.44,9.70,"9,070,000,000.00","10,507,000.00",58.60,7.61


In [19]:
# Review the complete dataset after adding the period classification
southafrica_morocco_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,tourist_arrivals,government_debt_pct_gdp,tourism_revenue_pct_gdp
0,2005,Morocco,"68,852,658,068.67",3.19,0.98,11.01,"5,426,000,000.00","6,077,000.00",54.80,7.88
1,2006,Morocco,"75,883,823,300.85",7.79,3.28,9.67,"6,900,000,000.00","6,777,000.00",50.60,9.09
2,2007,Morocco,"86,947,913,286.73",3.44,2.04,9.56,"8,307,000,000.00","7,701,000.00",47.10,9.55
3,2008,Morocco,"101,822,906,949.06",5.68,3.71,9.57,"8,885,000,000.00","8,209,000.00",42.00,8.73
4,2009,Morocco,"101,154,952,240.88",3.75,0.97,8.96,"7,980,000,000.00","8,661,000.00",42.60,7.89
5,2010,Morocco,"100,865,329,473.44",3.50,0.99,9.09,"8,176,000,000.00","9,752,000.00",45.30,8.11
6,2011,Morocco,"110,080,631,332.38",5.52,0.91,8.91,"9,101,000,000.00","9,784,000.00",48.60,8.27
7,2012,Morocco,"106,937,392,311.13",3.06,1.29,8.99,"8,491,000,000.00","9,830,000.00",52.30,7.94
8,2013,Morocco,"115,739,287,305.08",4.12,1.88,9.23,"8,201,000,000.00","10,349,000.00",57.10,7.09
9,2014,Morocco,"119,130,841,411.66",2.72,0.44,9.70,"9,070,000,000.00","10,507,000.00",58.60,7.61


In [20]:
# Create before/after year columns

# Years before the 2010 World Cup are labeled "Before"
# Years from 2010 onward are labeled "After"
southafrica_morocco_df["period"] = np.where(
    southafrica_morocco_df["year"] < 2010,
    "Before",
    "After"
)

southafrica_morocco_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,tourist_arrivals,government_debt_pct_gdp,tourism_revenue_pct_gdp,period
0,2005,Morocco,"68,852,658,068.67",3.19,0.98,11.01,"5,426,000,000.00","6,077,000.00",54.80,7.88,Before
1,2006,Morocco,"75,883,823,300.85",7.79,3.28,9.67,"6,900,000,000.00","6,777,000.00",50.60,9.09,Before
2,2007,Morocco,"86,947,913,286.73",3.44,2.04,9.56,"8,307,000,000.00","7,701,000.00",47.10,9.55,Before
3,2008,Morocco,"101,822,906,949.06",5.68,3.71,9.57,"8,885,000,000.00","8,209,000.00",42.00,8.73,Before
4,2009,Morocco,"101,154,952,240.88",3.75,0.97,8.96,"7,980,000,000.00","8,661,000.00",42.60,7.89,Before
5,2010,Morocco,"100,865,329,473.44",3.50,0.99,9.09,"8,176,000,000.00","9,752,000.00",45.30,8.11,After
6,2011,Morocco,"110,080,631,332.38",5.52,0.91,8.91,"9,101,000,000.00","9,784,000.00",48.60,8.27,After
7,2012,Morocco,"106,937,392,311.13",3.06,1.29,8.99,"8,491,000,000.00","9,830,000.00",52.30,7.94,After
8,2013,Morocco,"115,739,287,305.08",4.12,1.88,9.23,"8,201,000,000.00","10,349,000.00",57.10,7.09,After
9,2014,Morocco,"119,130,841,411.66",2.72,0.44,9.70,"9,070,000,000.00","10,507,000.00",58.60,7.61,After


In [21]:
# Calculate average indicators by country and period
# Select the indicators used to compare the Before and After periods
metrics = [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_pct",
    "government_debt_pct_gdp",
    "tourism_revenue_current_usd",
    "tourist_arrivals"
]

# Calculate the mean of each indicator for every country-period combination
avg_table = southafrica_morocco_df.pivot_table(
    index="country",
    columns="period",
    values=metrics,
    aggfunc="mean"
)

# Round the displayed results while keeping the underlying values unchanged
avg_table.round(2)

gdp_growth_pct        government_debt_pct_gdp         \
period                After Before                   After Before   
country                                                             
Morocco                3.88   4.77                   53.38  47.42   
South Africa           2.30   3.58                   38.70  26.58   

             inflation_pct        tourism_revenue_current_usd  \
period               After Before                       After   
country                                                         
Morocco               1.18   2.20            8,467,333,333.33   
South Africa          5.21   5.73           10,384,833,333.33   

                              tourist_arrivals              unemployment_pct  \
period                 Before            After       Before            After   
country                                                                        
Morocco      7,499,600,000.00    10,127,333.33 7,485,000.00             9.23   
South Africa 9,185,600,000.00    13,211,500.00 8,899,200.00            24.77   

                     
period       Before  
country              
Morocco        9.75  
South Africa  22.59

In [39]:
# Measure the change from before to after
# Extract the country averages for each period from the pivot table

#Cross sections (xs) works this way:
    #   dataframe.xs(
    #       label_to_find,
    #       level=where_to_find_it, --> in this case
    #        axis=rows_or_columns
    #   )

before = avg_table.xs(
    "Before",
    level=1,
    axis=1
)

after = avg_table.xs(
    "After",
    level=1,
    axis=1
)

# Positive values indicate an increase; negative values indicate a decrease
change_table = after - before

change_table.round(2)

,gdp_growth_pct,government_debt_pct_gdp,inflation_pct,tourism_revenue_current_usd,tourist_arrivals,unemployment_pct
country,,,,,,
Morocco,-0.89,5.96,-1.02,"967,733,333.33","2,642,333.33",-0.52
South Africa,-1.27,12.12,-0.52,"1,199,233,333.33","4,312,300.00",2.18


In [40]:
# Estimate relative world cup impact
# Subtract Morocco's change from South Africa's change to create a comparison group
impact_table = (
    change_table.loc["South Africa"]
    -
    change_table.loc["Morocco"]
)

# Convert the resulting Series into a clearly labelled one-column table
impact_table = impact_table.to_frame(
    name="South Africa_minus_Morocco"
)

impact_table.round(2)

,South Africa_minus_Morocco
gdp_growth_pct,-0.38
government_debt_pct_gdp,6.16
inflation_pct,0.50
tourism_revenue_current_usd,"231,500,000.00"
tourist_arrivals,"1,669,966.67"
unemployment_pct,2.71


In [41]:
# Create a GDP table with years as rows and countries as columns
# This makes it easy to compare South Africa and Morocco GDP year by year
gdp_table = southafrica_morocco_df.pivot_table(
    index="year",
    columns="country",
    values="gdp_current_usd",
    aggfunc="mean"
)

gdp_table

country,Morocco,South Africa
year,,
2005,"68,852,658,068.67","288,867,217,196.53"
2006,"75,883,823,300.85","303,858,675,363.64"
2007,"86,947,913,286.73","333,077,117,253.68"
2008,"101,822,906,949.06","316,131,258,616.31"
2009,"101,154,952,240.88","329,754,060,647.13"
2010,"100,865,329,473.44","417,363,822,801.71"
2011,"110,080,631,332.38","458,199,494,830.83"
2012,"106,937,392,311.13","434,400,545,085.81"
2013,"115,739,287,305.08","400,886,013,595.57"


In [42]:
# Create a tourism revenue table with years as rows and countries as columns
# This shows how much tourism revenue each country earned each year
tourism_revenue_table = southafrica_morocco_df.pivot_table(
    index="year",
    columns="country",
    values="tourism_revenue_current_usd",
    aggfunc="mean"
)

tourism_revenue_table

country,Morocco,South Africa
year,,
2005,"5,426,000,000.00","8,629,000,000.00"
2006,"6,900,000,000.00","9,211,000,000.00"
2007,"8,307,000,000.00","10,226,000,000.00"
2008,"8,885,000,000.00","9,178,000,000.00"
2009,"7,980,000,000.00","8,684,000,000.00"
2010,"8,176,000,000.00","10,309,000,000.00"
2011,"9,101,000,000.00","10,706,000,000.00"
2012,"8,491,000,000.00","11,202,000,000.00"
2013,"8,201,000,000.00","10,468,000,000.00"


In [43]:
# Create a tourist arrivals table with years as rows and countries as columns
# This compares the number of international tourist arrivals each year
tourist_arrivals_table = southafrica_morocco_df.pivot_table(
    index="year",
    columns="country",
    values="tourist_arrivals",
    aggfunc="mean"
)

tourist_arrivals_table

country,Morocco,South Africa
year,,
2005,"6,077,000.00","7,518,000.00"
2006,"6,777,000.00","8,509,000.00"
2007,"7,701,000.00","9,208,000.00"
2008,"8,209,000.00","9,729,000.00"
2009,"8,661,000.00","9,532,000.00"
2010,"9,752,000.00","11,303,000.00"
2011,"9,784,000.00","12,097,000.00"
2012,"9,830,000.00","13,069,000.00"
2013,"10,349,000.00","14,318,000.00"


In [44]:
# Create a GDP growth table with years as rows and countries as columns
# This compares each country's yearly GDP growth percentage
gdp_growth_table = southafrica_morocco_df.pivot_table(
    index="year",
    columns="country",
    values="gdp_growth_pct",
    aggfunc="mean"
)

gdp_growth_table

country,Morocco,South Africa
year,,
2005,3.19,5.28
2006,7.79,5.60
2007,3.44,5.36
2008,5.68,3.19
2009,3.75,-1.54
2010,3.50,3.04
2011,5.52,3.17
2012,3.06,2.40
2013,4.12,2.49


In [45]:
# Create a government debt table with years as rows and countries as columns
# This compares government debt as a percentage of GDP for each country
gvt_debt= southafrica_morocco_df.pivot_table(
    index="year",
    columns= ["country"],
    values="government_debt_pct_gdp",
)

gvt_debt

country,Morocco,South Africa
year,,
2005,54.80,29.60
2006,50.60,28.00
2007,47.10,24.30
2008,42.00,24.00
2009,42.60,27.00
2010,45.30,31.20
2011,48.60,34.70
2012,52.30,37.40
2013,57.10,40.40


In [46]:
# Create a tourism revenue percentage-of-GDP table by year and country
# This shows how important tourism revenue was relative to each country's GDP
tourism_pct_gdp = southafrica_morocco_df.pivot_table(
    index="year",
    columns= ["country"],
    values="tourism_revenue_pct_gdp",
)

tourism_pct_gdp

country,Morocco,South Africa
year,,
2005,7.88,2.99
2006,9.09,3.03
2007,9.55,3.07
2008,8.73,2.90
2009,7.89,2.63
2010,8.11,2.47
2011,8.27,2.34
2012,7.94,2.58
2013,7.09,2.61


In [47]:
# Function to find the percentage change between 2 years in a dataframe.

def calculate_percent_change(df, start_year, end_year, value_column):
    start_value = df.loc[df["year"] == start_year, value_column].iloc[0]
    end_value = df.loc[df["year"] == end_year, value_column].iloc[0]

    return ((end_value - start_value) / start_value) * 100

In [48]:
calculate_percent_change(southafrica_df,2005, 2015,"gdp_current_usd" )

np.float64(20.02393134928674)

In [49]:
# Web Scraping for 2010 World Cup images

from bs4 import BeautifulSoup

# Step 1 URL
url = "https://en.wikipedia.org/wiki/2010_FIFA_World_Cup"

# Step 2 User Agent Header
headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari 537.36"}

# Step 3a Make the Requests and save the Response
wiki_response = requests.get(url, headers=headers) 

In [50]:
#  Step 3b Test status code (Good practice)
if wiki_response.status_code == 200:
    print("Ok")
else:
    print("Not ok")

Ok


In [51]:
# Step 4 Parse the Response
wiki_soup = BeautifulSoup(wiki_response.content, "html.parser")

# Data Step 1 - Look at the HTML and search for all images. We're looking for the logo.
all_images = wiki_soup.find_all("img")
print(len(all_images))

708


In [52]:
# Verify the contents
from IPython.display import Image, display

for img in all_images:
    src = img.get("src")

    if src and "2010_FIFA_World_Cup.svg" in src:
        img_url = "https:" + src
        display(Image(url=img_url))